# Rally E2B Direct Full Compose

Consumes the Rally E2B prep artifact, exports the optimized direct Heretic text sessions, and composes the direct full text+image package by copying the reference Gemma4 q4f16 vision files. Upload is disabled by default so the existing public direct full repo is not replaced until the storage/private-state decision is clear.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(3):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'onnx>=1.19.0',
    'onnx-ir>=0.1.12',
    'onnxruntime>=1.23.0',
    'onnxscript>=0.5.0',
    'onnxconverter-common>=1.14.0',
    'sentencepiece>=0.2.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)

prep_candidates = [
    Path('/kaggle/input/rally-e2b-export-prep/rally-e2b-export-prep'),
    Path('/kaggle/input/rally-e2b-export-prep'),
    Path('/kaggle/input/notebooks/thomasjvu/rally-e2b-export-prep/rally-e2b-export-prep'),
    Path('/kaggle/input/notebooks/thomasjvu/rally-e2b-export-prep'),
]
prep_root = first_existing(prep_candidates)
if prep_root is None:
    checked = '\n'.join(str(path) for path in prep_candidates)
    raise FileNotFoundError(f'Could not locate prep kernel output. Checked:\n{checked}')

staged_repo = prep_root / 'heretic-to-onnx'
staged_template = prep_root / 'optimized-template'
if not (staged_repo / 'scripts/kaggle_rally_e2b_two_stage_export.py').exists():
    raise FileNotFoundError(staged_repo / 'scripts/kaggle_rally_e2b_two_stage_export.py')
for relative in ['onnx/decoder_model_merged_q4f16.onnx', 'onnx/vision_encoder_q4f16.onnx', 'onnx/vision_encoder_q4f16.onnx_data']:
    if not (staged_template / relative).exists():
        raise FileNotFoundError(staged_template / relative)

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(staged_repo, REPO_DIR)
print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('optimized_template=', staged_template)

WORK_DIR = Path(os.environ.get('RALLY_EXPORT_WORK_DIR', '/kaggle/working/rally-e2b-direct-full-compose'))
REPORT_PATH = WORK_DIR / 'rally-e2b-browser-export-report.json'
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_e2b_two_stage_export.py'),
    '--work-dir', str(WORK_DIR),
    '--report-path', str(REPORT_PATH),
    '--scratch-dir', os.environ.get('RALLY_EXPORT_SCRATCH_DIR', '/kaggle/temp/rally-e2b-direct-full-compose'),
    '--direct-full-repo', os.environ.get('RALLY_DIRECT_FULL_REPO', 'thomasjvu/rally-2b'),
    '--direct-text-repo', os.environ.get('RALLY_DIRECT_TEXT_REPO', 'thomasjvu/rally-2b-text'),
    '--rp-merged-repo', os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-2b-rp-a100-b75-merged'),
    '--rp-full-repo', os.environ.get('RALLY_RP_FULL_REPO', 'thomasjvu/rally-2b-rp'),
    '--rp-text-repo', os.environ.get('RALLY_RP_TEXT_REPO', 'thomasjvu/rally-2b-rp-text'),
    '--export-device', os.environ.get('RALLY_EXPORT_DEVICE', 'cpu'),
    '--optimized-template-dir', str(staged_template),
    '--full-package-mode', 'template',
    '--skip-rp',
    '--no-score',
]
if os.environ.get('RALLY_UPLOAD', '0') != '1':
    cmd.append('--no-upload')
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json, os

report_path = Path(os.environ.get('RALLY_EXPORT_WORK_DIR', '/kaggle/working/rally-e2b-direct-full-compose')) / 'rally-e2b-browser-export-report.json'
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'exports': [item.get('target', {}).get('repo_id') for item in report.get('exports', [])],
        'warnings': report.get('warnings', []),
    }, indent=2))
else:
    raise FileNotFoundError(report_path)
